# DQN (PyTorch Scaffolding)

Implement a minimal Deep Q-Network agent in PyTorch.

Focus areas:
- replay buffer sampling
- epsilon-greedy action selection
- Bellman target computation
- online/target network updates

## 1) Setup

In [1]:
import random
from collections import deque
from dataclasses import dataclass
from typing import Deque, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 2) Tiny environment (self-contained)

A short-horizon context-action task that is simple but enough for DQN wiring.

In [2]:
@dataclass
class TinyEnv:
    state_dim: int = 6
    action_dim: int = 3
    horizon: int = 20

    def reset(self):
        self.t = 0
        self.s = torch.randint(0, self.state_dim, (1,)).item()
        return self._obs(self.s)

    def _obs(self, idx: int):
        x = torch.zeros(self.state_dim, dtype=torch.float32)
        x[idx] = 1.0
        return x

    def step(self, action: int):
        # Context groups map to preferred actions
        # states 0,1 -> action 0; 2,3 -> action 1; 4,5 -> action 2
        target_action = self.s // 2
        reward = 1.0 if action == target_action else -0.25

        self.t += 1
        done = self.t >= self.horizon

        self.s = torch.randint(0, self.state_dim, (1,)).item()
        next_state = self._obs(self.s)
        return next_state, reward, done, {}

## 3) Replay buffer scaffold

In [ ]:
Transition = Tuple[torch.Tensor, int, float, torch.Tensor, bool]

class ReplayBuffer:
    def __init__(self, capacity: int = 50_000):
        self.buf: Deque[Transition] = deque(maxlen=capacity)

    def push(self, s, a, r, ns, done):
        # TODO: append transition to buffer
        self.buf.append((s, a, r, ns, done))
        # raise NotImplementedError

    def sample(self, batch_size: int):
        # TODO:
        # 1) random sample transitions
        # 2) stack into tensors
        # 3) return s, a, r, ns, done
        # raise NotImplementedError

        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, done = zip(*batch)

        s = torch.stack(s).to(device=device, dtype=torch.float32)            # [B, state_dim]
        a = torch.tensor(a, device=device, dtype=torch.long)                 # [B]
        r = torch.tensor(r, device=device, dtype=torch.float32)              # [B]
        ns = torch.stack(ns).to(device=device, dtype=torch.float32)          # [B, state_dim]
        done = torch.tensor(done, device=device, dtype=torch.float32)        # [B], 0/1 mask
        return s, a, r, ns, done

    def __len__(self):
        return len(self.buf)

## 4) Q-network scaffold

In [4]:
class QNet(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x: torch.Tensor):
        # TODO: return Q-values with shape [B, action_dim] (or [action_dim] for single state)
        return self.net(x)
        # raise NotImplementedError

## 5) Epsilon-greedy policy scaffold

In [5]:
def select_action(qnet: QNet, state: torch.Tensor, eps: float, action_dim: int):
    # TODO:
    # if random.random() < eps: return random action
    # else: return argmax_a Q(state, a)
    # raise NotImplementedError
    if random.random() < eps:
        return random.randint(0, action_dim)
    else:
        qs = qnet(state)
        return torch.argmax(qs)

## 6) DQN update step scaffold

Implement one gradient step using:
`target = r + gamma * (1 - done) * max_a' Q_target(next_state, a')`

In [ ]:
def dqn_update(
    qnet: QNet,
    target_qnet: QNet,
    optimizer: torch.optim.Optimizer,
    batch,
    gamma: float = 0.99,
):
    # batch = (s, a, r, ns, done)
    # TODO:
    # 1) compute q_sa from online net for chosen actions
    # 2) compute target with target net (no grad)
    # 3) mse/huber loss, backward, optimizer step
    # 4) return scalar loss
    raise NotImplementedError

## 7) Training loop scaffold

Key pieces to implement:
- environment interaction and replay insertion
- warmup before updates
- periodic target network sync
- epsilon decay schedule

In [ ]:
def train_dqn(
    episodes: int = 300,
    batch_size: int = 64,
    gamma: float = 0.99,
    lr: float = 1e-3,
    start_eps: float = 1.0,
    end_eps: float = 0.05,
    eps_decay: float = 0.995,
    target_sync_every: int = 20,
    warmup_steps: int = 200,
):
    env = TinyEnv()
    qnet = QNet(env.state_dim, env.action_dim).to(device)
    target_qnet = QNet(env.state_dim, env.action_dim).to(device)
    target_qnet.load_state_dict(qnet.state_dict())

    optimizer = torch.optim.Adam(qnet.parameters(), lr=lr)
    replay = ReplayBuffer(capacity=50_000)

    eps = start_eps
    total_steps = 0
    ep_returns: List[float] = []

    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_return = 0.0

        while not done:
            # TODO: action selection
            # action = select_action(qnet, state, eps, env.action_dim)

            # TODO: environment step and replay push
            # next_state, reward, done, _ = env.step(action)
            # replay.push(state, action, reward, next_state, done)

            # TODO: update online network after warmup
            # if total_steps > warmup_steps and len(replay) >= batch_size:
            #     batch = replay.sample(batch_size)
            #     loss = dqn_update(qnet, target_qnet, optimizer, batch, gamma)

            # TODO: periodically sync target net
            # if total_steps % target_sync_every == 0:
            #     target_qnet.load_state_dict(qnet.state_dict())

            # bookkeeping
            # state = next_state
            # ep_return += reward
            # total_steps += 1
            raise NotImplementedError

        # TODO: epsilon decay
        # eps = max(end_eps, eps * eps_decay)

        ep_returns.append(ep_return)
        if (ep + 1) % 50 == 0:
            avg = sum(ep_returns[-50:]) / 50
            print(f"episode {ep+1:4d} | avg_return(last50)={avg:.3f} | eps={eps:.3f}")

    return qnet, ep_returns

## 8) Smoke checks (uncomment after TODOs)

In [ ]:
# env = TinyEnv()
# q = QNet(env.state_dim, env.action_dim)
# s = env.reset()
#
# qvals = q(s)
# print("qvals shape:", tuple(qvals.shape))
#
# rb = ReplayBuffer(capacity=100)
# a = 0
# ns, r, d, _ = env.step(a)
# rb.push(s, a, r, ns, d)
# print("replay size:", len(rb))
#
# model, hist = train_dqn(episodes=300)
# print("final avg return (last 20):", sum(hist[-20:]) / 20)

## 9) Stretch prompts

- Switch to Huber loss (`F.smooth_l1_loss`) and compare stability.
- Implement Double DQN targets.
- Implement soft target updates (Polyak averaging).
- Add prioritized replay and compare sample efficiency.